## Kỹ thuật Principal Component Analysis (PCA)

## Import

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Load dữ liệu

In [67]:
X_train = pd.read_csv(
    "../data/processed/X_train_processed.csv"
)

X_val = pd.read_csv(
    "../data/processed/X_val_processed.csv"
)

X_test = pd.read_csv(
    "../data/processed/X_test_processed.csv"
)

y_train = pd.read_csv(
    "../data/processed/y_train.csv"
).squeeze()

y_val = pd.read_csv(
    "../data/processed/y_val.csv"
).squeeze()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

X_train: (1168, 239)
X_val: (292, 239)
X_test: (1459, 239)
y_train: (1168,)
y_val: (292,)


## Kiểm tra NaN

In [68]:
print("NaN X_train:", X_train.isna().sum().sum())
print("NaN X_val:", X_val.isna().sum().sum())
print("NaN X_test:", X_test.isna().sum().sum())

NaN X_train: 0
NaN X_val: 0
NaN X_test: 5


## Xử lý trước PCA

In [69]:
imputer = SimpleImputer(
    strategy="median"
)

X_train_imputed = imputer.fit_transform(
    X_train
)

X_val_imputed = imputer.transform(
    X_val
)

X_test_imputed = imputer.transform(
    X_test
)

In [70]:
scaler_pca = StandardScaler()

X_train_scaled = scaler_pca.fit_transform(
    X_train_imputed
)

X_val_scaled = scaler_pca.transform(
    X_val_imputed
)

X_test_scaled = scaler_pca.transform(
    X_test_imputed
)

## StandardScaler

In [55]:
pca = PCA(
    n_components=0.95,
    random_state=42
)

X_train_pca = pca.fit_transform(
    X_train_scaled
)

X_val_pca = pca.transform(
    X_val_scaled
)

X_test_pca = pca.transform(
    X_test_scaled
)

## PCA 95%

In [ ]:
pca = PCA(
    n_components=0.95,
    random_state=42
)

X_train_pca = pca.fit_transform(
    X_train_scaled
)

X_val_pca = pca.transform(
    X_val_scaled
)

X_test_pca = pca.transform(
    X_test_scaled
)

## Kiểm tra PCA

In [ ]:
print(
    "Số feature ban đầu:",
    X_train.shape[1]
)

print(
    "Số component sau PCA:",
    X_train_pca.shape[1]
)

print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)

Số feature ban đầu: 239
Số component sau PCA: 145
Explained variance: 0.9511762757851808


## Explained Variance

In [71]:
explained_variance = pd.DataFrame({
    "Component": [
        f"PC{i+1}"
        for i in range(
            len(pca.explained_variance_ratio_)
        )
    ],
    "Explained Variance": pca.explained_variance_ratio_,
    "Cumulative Variance": np.cumsum(
        pca.explained_variance_ratio_
    )
})

explained_variance.head(20)

,Component,Explained Variance,Cumulative Variance
0,PC1,0.069270,0.069270
1,PC2,0.032959,0.102229
2,PC3,0.028185,0.130415
3,PC4,0.022398,0.152813
4,PC5,0.020232,0.173044
5,PC6,0.018591,0.191636
6,PC7,0.016587,0.208223
7,PC8,0.014876,0.223099
8,PC9,0.014394,0.237494
9,PC10,0.014238,0.251731


## Chuyển PCA thành DATAFRAME

In [ ]:
pca_features = [
    f"PC{i+1}"
    for i in range(
        X_train_pca.shape[1]
    )
]

X_train_pca = pd.DataFrame(
    X_train_pca,
    columns=pca_features
)

X_val_pca = pd.DataFrame(
    X_val_pca,
    columns=pca_features
)

X_test_pca = pd.DataFrame(
    X_test_pca,
    columns=pca_features
)

print("X_train_pca:", X_train_pca.shape)
print("X_val_pca:", X_val_pca.shape)
print("X_test_pca:", X_test_pca.shape)

(1168, 145)
(292, 145)
(1459, 145)


## Lưu kết quả FS

In [72]:
os.makedirs(
    "../data/pca",
    exist_ok=True
)

X_train_pca.to_csv(
    "../data/pca/X_train.csv",
    index=False
)

X_val_pca.to_csv(
    "../data/pca/X_val.csv",
    index=False
)

X_test_pca.to_csv(
    "../data/pca/X_test.csv",
    index=False
)

print("Đã lưu dữ liệu sau PCA.")

Đã lưu dữ liệu sau PCA.


## Khai báo model

In [ ]:
models = {
    "SVR": SVR(),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        random_state=42
    )
}

## Validation

In [73]:
results = []

for name, model in models.items():

    model.fit(
        X_train_pca,
        y_train
    )

    pred = model.predict(
        X_val_pca
    )

    mae = mean_absolute_error(
        y_val,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            pred
        )
    )

    r2 = r2_score(
        y_val,
        pred
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

## Kết quả validation

In [ ]:
results_df = pd.DataFrame(results)

results_df

,Model,MAE,RMSE,R2
0,SVR,59541.245440,88645.313679,-0.024466
1,Random Forest,19265.646575,33304.941098,0.855388
2,XGBoost,19469.527344,37448.445201,0.817167


## Gộp train + validation

In [74]:
X_full_original = pd.concat([
    X_train,
    X_val
], axis=0)

X_full_original = X_full_original.reset_index(
    drop=True
)

y_full = pd.concat([
    y_train.reset_index(drop=True),
    y_val.reset_index(drop=True)
], axis=0)

y_full = y_full.reset_index(
    drop=True
)

print("X_full:", X_full_original.shape)
print("y_full:", y_full.shape)

X_full: (1460, 239)
y_full: (1460,)


## Imputer final

In [ ]:
imputer_final = SimpleImputer(
    strategy="median"
)

X_full_imputed = imputer_final.fit_transform(
    X_full_original
)

X_test_imputed = imputer_final.transform(
    X_test
)

## Scaler final

In [75]:
scaler_final = StandardScaler()

X_full_scaled = scaler_final.fit_transform(
    X_full_imputed
)

X_test_scaled = scaler_final.transform(
    X_test_imputed
)

## PCA final

In [76]:
pca_final = PCA(
    n_components=0.95,
    random_state=42
)

X_full_pca = pca_final.fit_transform(
    X_full_scaled
)

X_test_final_pca = pca_final.transform(
    X_test_scaled
)

## Chuyển PCA final thành DATAFRAME

In [77]:
pca_features_final = [
    f"PC{i+1}"
    for i in range(
        X_full_pca.shape[1]
    )
]

X_full_pca = pd.DataFrame(
    X_full_pca,
    columns=pca_features_final
)

X_test_final_pca = pd.DataFrame(
    X_test_final_pca,
    columns=pca_features_final
)

print(
    "X_full_pca:",
    X_full_pca.shape
)

print(
    "X_test_final_pca:",
    X_test_final_pca.shape
)

print(
    "Final explained variance:",
    pca_final.explained_variance_ratio_.sum()
)

X_full_pca: (1460, 147)
X_test_final_pca: (1459, 147)
Final explained variance: 0.9511648364294636


## Load test 

In [78]:
test = pd.read_csv(
    "../data/test.csv"
)

In [79]:
os.makedirs(
    "../data/pca/submission",
    exist_ok=True
)

for name, model in models.items():

    model.fit(
        X_full_pca,
        y_full
    )

    pred_test = model.predict(
        X_test_final_pca
    )

    submission = pd.DataFrame({
        "Id": test["Id"],
        "SalePrice": pred_test
    })

    submission.to_csv(
        f"../data/pca/submission/submission_{name}.csv",
        index=False
    )

    print(
        f"Đã lưu submission cho {name}"
    )

Đã lưu submission cho SVR
Đã lưu submission cho Random Forest
Đã lưu submission cho XGBoost
